# GPU performance basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoaibphysics/custom-kernels/blob/main/notebooks/01_performance_basics.ipynb)

What determines how quickly a computation can run?

We explore the relationship between arithmetic, memory traffic,
and execution time. Starting with vector addition, we will count
operations and bytes, build simple performance estimates, and
later compare those estimates with measurements.

## 1. Counting operations and bytes

Consider two vectors, $x$ and $y$, each containing $N$ elements.
Their elementwise sum is

$$
z_i = x_i + y_i,
\qquad i = 0, 1, \ldots, N-1.
$$

Each output element requires one floating-point addition.
We count this as one floating-point operation, or FLOP.

FLOPs describe an amount of arithmetic.
FLOP/s describes a rate: how many floating-point operations
are performed per second.

The arithmetic also requires data. For each output element,
we must:

1. Read one element from $x$.
2. Read one element from $y$.
3. Write one element to $z$.

Assume all three vectors use FP32, meaning each element occupies
32 bits, or 4 bytes. Reading the two inputs therefore accounts
for 8 bytes, and writing the output accounts for another 4 bytes.

For $N$ elements, our counts are

$$
F = N \quad \text{FLOPs},
$$

$$
Q = N(4 + 4 + 4) = 12N \quad \text{bytes}.
$$

This is a simple count of logical memory traffic. We assume
each input is read once and each output is written once.
We initially ignore caches and other hardware effects.

The arrays are assumed to already reside on the device.
Transfers between the CPU and GPU are outside this estimate.

In [1]:
n_elements = 1_000_000
bytes_per_element = 4  # FP32

# One addition for each output element.
total_flops = n_elements

# Read two input arrays and write one output array.
bytes_read = 2 * n_elements * bytes_per_element
bytes_written = n_elements * bytes_per_element
total_bytes = bytes_read + bytes_written

print(f"Elements: {n_elements:,}")
print(f"Arithmetic: {total_flops:,} FLOPs")
print(f"Input reads: {bytes_read / 1e6:.2f} MB")
print(f"Output writes: {bytes_written / 1e6:.2f} MB")
print(f"Total logical traffic: {total_bytes / 1e6:.2f} MB")

Elements: 1,000,000
Arithmetic: 1,000,000 FLOPs
Input reads: 8.00 MB
Output writes: 4.00 MB
Total logical traffic: 12.00 MB


## 2. Arithmetic intensity

Arithmetic intensity measures how much arithmetic a computation
performs relative to the amount of data it moves:

$$
I = \frac{F}{Q},
$$

where $F$ is the number of floating-point operations and $Q$ is
the number of bytes moved.

Its unit is FLOP per byte.

For our FP32 vector addition:

$$
I = \frac{N}{12N}
  = \frac{1}{12}
  \approx 0.0833 \text{ FLOP/byte}.
$$

The factor $N$ cancels. Increasing the vector length increases
both the arithmetic and the memory traffic proportionally.

Under this model, we move 12 bytes for every addition. This
suggests that data movement may be an important performance
constraint.

Arithmetic intensity describes the workload. It is not a
measurement of execution speed. To estimate execution time,
we also need information about the hardware.

In [2]:
arithmetic_intensity = total_flops / total_bytes
bytes_per_flop = total_bytes / total_flops

print(f"Arithmetic intensity: {arithmetic_intensity:.6f} FLOP/byte")
print(f"Data movement per FLOP: {bytes_per_flop:.2f} bytes/FLOP")

Arithmetic intensity: 0.083333 FLOP/byte
Data movement per FLOP: 12.00 bytes/FLOP


## 3. From memory traffic to execution time

Memory bandwidth describes how many bytes a memory system can
transfer per second.

If an operation transfers $Q$ bytes through a memory interface
with bandwidth $B$, its ideal transfer time is

$$
t_{\mathrm{memory}} = \frac{Q}{B}.
$$

The units are

$$
\frac{\text{bytes}}{\text{bytes/second}} = \text{seconds}.
$$

Consider a fictional GPU with memory bandwidth of 500 GB/s.
We use decimal units:

$$
500 \text{ GB/s} = 500 \times 10^9 \text{ bytes/second}.
$$

For this simplified model, assume all the bytes we counted must
pass through GPU main memory. We ignore caching and assume the
operation can fully utilize the stated bandwidth.

This gives an optimistic memory-time estimate. It is not a
measurement: actual execution also depends on launch overhead,
arithmetic, access patterns, and other hardware effects.

In [3]:
n_elements = 1_000_000
bytes_per_element = 4  # FP32

# Two input reads and one output write per element.
total_bytes = 3 * n_elements * bytes_per_element

# Fictional hardware specification, not a measured value.
bandwidth_gb_per_second = 500
bandwidth_bytes_per_second = bandwidth_gb_per_second * 1e9

memory_time_seconds = total_bytes / bandwidth_bytes_per_second
memory_time_microseconds = memory_time_seconds * 1e6

print(f"Logical memory traffic: {total_bytes / 1e6:.2f} MB")
print(f"Assumed bandwidth: {bandwidth_gb_per_second} GB/s")
print(f"Ideal memory time: {memory_time_microseconds:.2f} microseconds")

Logical memory traffic: 12.00 MB
Assumed bandwidth: 500 GB/s
Ideal memory time: 24.00 microseconds


## 4. Comparing arithmetic and memory limits

Let:

- $F$ be the arithmetic work, measured in FLOPs.
- $P$ be the applicable peak arithmetic throughput, in FLOP/s.
- $Q$ be the memory traffic, measured in bytes.
- $B$ be the memory bandwidth, in bytes/s.

The ideal time required by each resource is

$$
t_{\mathrm{compute}} = \frac{F}{P},
\qquad
t_{\mathrm{memory}} = \frac{Q}{B}.
$$

A computation must satisfy both constraints. Therefore,

$$
t_{\mathrm{execution}}
\geq
\max\left(\frac{F}{P}, \frac{Q}{B}\right).
$$

We take the maximum because arithmetic and memory transfers can
overlap across the workload. Adding the two times would assume
they happen entirely sequentially.

The maximum is an optimistic lower bound, not a promise of perfect
overlap or full hardware utilization.

If the memory term is larger, the model predicts a memory-bound
operation. If the arithmetic term is larger, it predicts a
compute-bound operation.

Using arithmetic intensity $I = F/Q$, we can also express the
performance ceiling as

$$
\text{attainable FLOP/s} \leq \min(P, B I).
$$

This is the basic roofline model.

In [4]:
# Workload: one million FP32 additions.
n_elements = 1_000_000
bytes_per_element = 4

total_flops = n_elements
total_bytes = 3 * n_elements * bytes_per_element

# Fictional hardware.
peak_flops_per_second = 10e12       # 10 TFLOP/s
bandwidth_bytes_per_second = 500e9  # 500 GB/s

compute_time = total_flops / peak_flops_per_second
memory_time = total_bytes / bandwidth_bytes_per_second

time_lower_bound = max(compute_time, memory_time)
throughput_ceiling = total_flops / time_lower_bound

print(f"Ideal compute time: {compute_time * 1e6:.2f} microseconds")
print(f"Ideal memory time: {memory_time * 1e6:.2f} microseconds")
print(f"Execution-time lower bound: {time_lower_bound * 1e6:.2f} microseconds")
print(f"Throughput ceiling: {throughput_ceiling / 1e9:.2f} GFLOP/s")

Ideal compute time: 0.10 microseconds
Ideal memory time: 24.00 microseconds
Execution-time lower bound: 24.00 microseconds
Throughput ceiling: 41.67 GFLOP/s


## 5. Interpreting measured performance

A kernel's arithmetic throughput is

$$
P_{\mathrm{achieved}} = \frac{F}{t},
$$

where $t$ is its measured execution time.

For a memory-bound operation, another useful quantity is
effective memory bandwidth:

$$
B_{\mathrm{effective}} = \frac{Q}{t}.
$$

Here, $Q$ is our logical byte count. Effective bandwidth describes
how quickly the operation processes that amount of data.

Consider our one-million-element FP32 addition. Suppose a
correctly timed GPU execution took 30 microseconds. Then

$$
B_{\mathrm{effective}}
=
\frac{12 \times 10^6}{30 \times 10^{-6}}
=
400 \text{ GB/s}.
$$

Its arithmetic throughput would be approximately 33.3 GFLOP/s.

Compared with our fictional GPU's 500 GB/s memory-bandwidth
ceiling, this represents 80% of the assumed bandwidth. The low
arithmetic throughput does not, by itself, indicate an inefficient
kernel.

This percentage is not overall GPU utilization.

Effective bandwidth uses logical traffic, which can differ from
actual main-memory traffic because of caching. Launch overhead
can also dominate small workloads.

The following calculation uses a hypothetical execution time.
It is an illustration, not a GPU benchmark.

In [5]:
# Same workload: one million FP32 additions.
n_elements = 1_000_000
total_flops = n_elements
total_bytes = 3 * n_elements * 4

# Hypothetical timing and hardware specification.
example_time_us = 30.0
example_time_seconds = example_time_us * 1e-6
assumed_bandwidth_bytes_per_second = 500e9

effective_bandwidth = total_bytes / example_time_seconds
achieved_throughput = total_flops / example_time_seconds

bandwidth_fraction = effective_bandwidth / assumed_bandwidth_bytes_per_second

print("Illustrative calculation — no GPU measurement performed")
print(f"Assumed execution time: {example_time_us:.2f} microseconds")
print(f"Effective bandwidth: {effective_bandwidth / 1e9:.2f} GB/s")
print(f"Arithmetic throughput: {achieved_throughput / 1e9:.2f} GFLOP/s")
print(f"Fraction of assumed bandwidth: {bandwidth_fraction:.1%}")

Illustrative calculation — no GPU measurement performed
Assumed execution time: 30.00 microseconds
Effective bandwidth: 400.00 GB/s
Arithmetic throughput: 33.33 GFLOP/s
Fraction of assumed bandwidth: 80.0%


## 6. Why fusion helps

Consider the computation

$$
z_i = \operatorname{ReLU}(x_i + y_i),
$$

where

$$
\operatorname{ReLU}(a) = \max(a, 0).
$$

ReLU preserves positive values and replaces negative values
with zero.

### Separate operations

A two-kernel implementation first computes an intermediate array:

$$
u_i = x_i + y_i.
$$

A second kernel reads that array and applies ReLU:

$$
z_i = \max(u_i, 0).
$$

For FP32 arrays, the logical memory traffic per element is:

| Operation | Reads | Writes | Total |
|---|---:|---:|---:|
| Addition | x and y: 8 bytes | u: 4 bytes | 12 bytes |
| ReLU | u: 4 bytes | z: 4 bytes | 8 bytes |
| Combined | | | 20 bytes |

Thus,

$$
Q_{\mathrm{separate}} = 20N \text{ bytes}.
$$

### Fused operation

A fused kernel reads an element from each input, computes the sum,
applies ReLU, and writes the final result.

The intermediate sum can remain in a register instead of being
written to an intermediate array and read by another kernel.

The logical traffic becomes

$$
Q_{\mathrm{fused}} = 12N \text{ bytes}.
$$

We avoid one intermediate write and one intermediate read:
8 bytes per element.

Assuming both implementations are memory-bound and achieve the
same bandwidth, their ideal memory-time ratio is

$$
\frac{t_{\mathrm{separate}}}{t_{\mathrm{fused}}}
=
\frac{20N/B}{12N/B}
=
\frac{5}{3}
\approx 1.67.
$$

This is a simplified prediction, not a measured speedup.
It ignores caching and launch overhead, and assumes the
intermediate array is not needed elsewhere.

In [6]:
n_elements = 1_000_000
bytes_per_element = 4  # FP32
bandwidth_bytes_per_second = 500e9

# Separate: read x, read y, write u, read u, write z.
separate_bytes = 5 * n_elements * bytes_per_element

# Fused: read x, read y, write z.
fused_bytes = 3 * n_elements * bytes_per_element

separate_memory_time = separate_bytes / bandwidth_bytes_per_second
fused_memory_time = fused_bytes / bandwidth_bytes_per_second

traffic_reduction = 1 - fused_bytes / separate_bytes
memory_time_ratio = separate_memory_time / fused_memory_time

print("Analytical estimates — no GPU measurement performed")
print(f"Separate operations: {separate_bytes / 1e6:.2f} MB")
print(f"Fused operation: {fused_bytes / 1e6:.2f} MB")
print(f"Traffic reduction: {traffic_reduction:.1%}")
print(f"Separate ideal memory time: {separate_memory_time * 1e6:.2f} microseconds")
print(f"Fused ideal memory time: {fused_memory_time * 1e6:.2f} microseconds")
print(f"Ideal memory-time ratio: {memory_time_ratio:.2f}x")

Analytical estimates — no GPU measurement performed
Separate operations: 20.00 MB
Fused operation: 12.00 MB
Traffic reduction: 40.0%
Separate ideal memory time: 40.00 microseconds
Fused ideal memory time: 24.00 microseconds
Ideal memory-time ratio: 1.67x


## 7. Why small workloads behave differently

Our roofline model considers arithmetic and memory traffic:

$$
t_{\mathrm{work}}
=
\max\left(\frac{F}{P}, \frac{Q}{B}\right).
$$

As the input becomes smaller, both terms shrink. However,
invoking a GPU operation also involves overhead, such as
dispatching and scheduling the work.

For an illustrative model of an isolated call, introduce
a fixed overhead $H$:

$$
t_{\mathrm{model}}
=
H + \max\left(\frac{F}{P}, \frac{Q}{B}\right).
$$

This assumes the overhead is not hidden by overlap.
Real execution can overlap CPU dispatch and GPU work, so this
is a teaching model rather than a universal timing formula.

Suppose our fictional overhead is 5 microseconds.

For a tiny workload, the arithmetic and memory times may be
much smaller than 5 microseconds. Reducing those times further
then makes little difference to the total.

For a large workload, the useful work takes longer, and the
fixed overhead becomes a smaller fraction of the total.

This explains why effective bandwidth can increase with input
size even when arithmetic intensity stays unchanged.

Fusion can help in two ways:

- Avoiding intermediate memory traffic.
- Reducing the number of kernel launches.

For small workloads, avoiding an additional launch may matter
more than saving a small amount of memory traffic.

In [7]:
# Fictional hardware and fixed overhead.
peak_flops_per_second = 10e12
bandwidth_bytes_per_second = 500e9
overhead_seconds = 5e-6

sizes = [1, 1_000, 100_000, 1_000_000, 10_000_000]

print("Illustrative model — not measured GPU timings")
print(
    f"{'Elements':>12} "
    f"{'Work (us)':>12} "
    f"{'Total (us)':>12} "
    f"{'Effective GB/s':>16}"
)

for n_elements in sizes:
    total_flops = n_elements
    total_bytes = 12 * n_elements  # FP32 addition

    compute_time = total_flops / peak_flops_per_second
    memory_time = total_bytes / bandwidth_bytes_per_second
    work_time = max(compute_time, memory_time)

    modeled_time = overhead_seconds + work_time
    effective_bandwidth = total_bytes / modeled_time

    print(
        f"{n_elements:>12,} "
        f"{work_time * 1e6:>12.6f} "
        f"{modeled_time * 1e6:>12.6f} "
        f"{effective_bandwidth / 1e9:>16.6f}"
    )

Illustrative model — not measured GPU timings
    Elements    Work (us)   Total (us)   Effective GB/s
           1     0.000024     5.000024         0.002400
       1,000     0.024000     5.024000         2.388535
     100,000     2.400000     7.400000       162.162162
   1,000,000    24.000000    29.000000       413.793103
  10,000,000   240.000000   245.000000       489.795918


## 8. Results and observations

### Arithmetic and memory traffic

For vectors containing N FP32 elements, elementwise addition
performs N additions. Reading two inputs and writing one output
requires 12N bytes of logical memory traffic, assuming each value
is accessed once.

The arithmetic intensity is therefore 1/12 FLOP per byte.
Increasing N increases arithmetic and traffic proportionally,
leaving this ratio unchanged.

### Choosing a performance metric

Low FLOP/s alone does not establish that an addition kernel is
inefficient. Addition performs little arithmetic per byte, so
memory bandwidth can limit its throughput.

Effective bandwidth, Q/t, describes how quickly the kernel
processes its logical memory traffic. It should be interpreted
alongside execution time, caching behavior, and the timing method.

### Kernel fusion

Computing ReLU(x + y) using separate kernels writes an intermediate
array and then reads it again. Fusion can keep each intermediate
sum inside the kernel, avoiding those two memory accesses while
preserving the addition and ReLU calculations.

Our FP32 traffic estimate decreases from 20N to 12N bytes:
a 40% reduction. The corresponding ideal memory-time ratio is
approximately 1.67, but this is not a guaranteed measured speedup.

### Small and large workloads

In our model, a fixed overhead dominates the execution time of
small operations. Doubling memory bandwidth makes little difference
when memory transfer accounts for only a small fraction of the
total time.

As the array grows, the fixed overhead becomes less significant.
Modeled effective bandwidth approaches the assumed hardware
bandwidth, even though arithmetic intensity remains unchanged.

### Limits of our conclusions

This notebook contains analytical estimates and hypothetical
timings, not GPU benchmarks. Arithmetic throughput, memory
bandwidth, and fixed overhead were assumed hardware parameters.

Real measurements must account for asynchronous execution,
compilation, caching, and other effects. We would need to execute
the actual operation, verify its result, and measure its completion
correctly before claiming a performance result.

### What I want to test next

For FP32 vector addition, I predict that effective bandwidth will
initially increase with array size as overhead becomes less
significant, eventually approaching a sustained rate.

I will test this by keeping the GPU and dtype fixed, sweeping
array sizes, and using a consistent timing method. I will also
consider whether caching explains departures from this prediction.